# Chapter 0 — Survey catalogues and the data of this tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/halo-substructure-tutorial/blob/main/notebooks/00_data.ipynb)

**Goal.** Learn what a row of an imaging-survey catalogue contains, how the data of this tutorial were
extracted, and take a first look at a Milky Way satellite: the dwarf galaxy Boötes I, discovered in the
Sloan Digital Sky Survey (SDSS) by Belokurov et al. (2006, https://ui.adsabs.harvard.edu/abs/2006ApJ...647L.111B).

**You will learn**
1. where the data come from: the WSDB database, q3c cone searches, and the files shipped with this tutorial;
2. what PSF magnitudes, their errors and the Galactic extinction are, and how to correct for extinction;
3. how to measure the depth of a catalogue from its magnitude errors;
4. that a satellite of the Milky Way is, in a catalogue, nothing more than a modest excess of faint stars.

**Units.** Angles in degrees (sometimes arcmin, marked ′), magnitudes in the AB-like SDSS system.

## 0.1 Where the data come from

Large surveys publish catalogues with 10⁸–10⁹ rows. Astronomers usually query them in a database. The data
of this tutorial were extracted from **WSDB** (the Whole Sky Database, Cambridge), a PostgreSQL server that holds
SDSS, Pan-STARRS1, the Legacy Surveys, DES, HSC, Gaia and ~150 other catalogues. Sky positions are indexed with
**q3c** (Koposov & Bartunov 2006, https://ui.adsabs.harvard.edu/abs/2006ASPC..351..735K), so that a cone search

```sql
SELECT objid, ra, dec, psfmag_g, psfmag_r, ... FROM sdssdr9.phototag
WHERE q3c_radial_query(ra, dec, 210.02, 14.5135, 2.0)   -- all objects within 2 deg of Boötes I
  AND mode = 1 AND type = 6 AND psfmag_g < 23           -- primary detections, point sources, g < 23
```

returns in seconds instead of scanning the whole table. You do not need database access: the script
`scripts/build_data.py` in the repository ran such queries once and saved small sky patches as Parquet files,
attached to a release of the repository. The function below downloads a file once and checks its SHA-256
checksum against the manifest written by that script.

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = 'https://github.com/vasilybelokurov/halo-substructure-tutorial/releases/download/data-v0.1/'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def load(name, cache=Path('data')):
    """Download <name>.parquet from the tutorial's data release once, check its SHA-256, return a DataFrame."""
    path = cache / f'{name}.parquet'
    if not path.exists():
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    sha = hashlib.sha256(path.read_bytes()).hexdigest()
    assert sha == MANIFEST[name]['sha256'], f'{path} does not match the manifest: delete it and run again'
    return pd.read_parquet(path)


def tangent(ra, dec, ra0, dec0):
    """Gnomonic (tangent-plane) projection about (ra0, dec0). Returns (xi, eta) in degrees: xi towards east, eta north."""
    ra, dec, ra0, dec0 = map(np.radians, (ra, dec, ra0, dec0))
    cosc = np.sin(dec0) * np.sin(dec) + np.cos(dec0) * np.cos(dec) * np.cos(ra - ra0)
    xi = np.cos(dec) * np.sin(ra - ra0) / cosc
    eta = (np.cos(dec0) * np.sin(dec) - np.sin(dec0) * np.cos(dec) * np.cos(ra - ra0)) / cosc
    return np.degrees(xi), np.degrees(eta)


print(f'{len(MANIFEST)} files in the data release:')
for k, v in MANIFEST.items():
    print(f'  {k:18s} {v["rows"]:>9,} rows  {v["bytes"] / 1e6:6.1f} MB')

The manifest also stores the exact SQL of every file — the selection is part of the science:

In [ ]:
print(MANIFEST['bootes1_sdss']['queries'][0])

## 0.2 What is in a row

Each row is one *primary* detection (`mode = 1`: the best of possibly several overlapping observations) that
the SDSS pipeline classified as a point source (`type = 6`; Chapter 1 shows how good that classification is).
For each of the g, r, i bands we have the **PSF magnitude** (the flux measured by fitting the point-spread
function), its error, and the **extinction** `extinction_g` = $A_g$, the dimming by Galactic dust in magnitudes,
from the Schlegel, Finkbeiner & Davis (1998) map times an SDSS band coefficient. Extinction-corrected
("dereddened") magnitudes are $g_0 = g - A_g$; colours are corrected the same way,
$(g-r)_0 = (g-r) - (A_g - A_r)$.

In [ ]:
ra0, dec0 = 210.0200, 14.5135                       # Boötes I, Local Volume Database v1.1.1 (Pace 2025)
cat = load('bootes1_sdss')
lvdb = load('lvdb').set_index('key')
print(cat.head().to_string())
g0 = cat.psfmag_g - cat.extinction_g
r0 = cat.psfmag_r - cat.extinction_r
gr0 = g0 - r0
print(f'\n{len(cat):,} point sources; median A_g = {cat.extinction_g.median():.3f} mag, '
      f'E(B-V) = A_g / 3.793 = {cat.extinction_g.median() / 3.793:.3f}')

## 0.3 How deep is the catalogue?

A magnitude error $\sigma_m$ corresponds to a signal-to-noise ratio $S/N = 2.5 / (\ln 10\, \sigma_m) \approx 1.086/\sigma_m$.
A common definition of depth is the magnitude where the median $S/N = 10$, i.e. $\sigma_m = 0.1086$ mag.
The catalogue also stops at g = 23 because we cut it there.

In [ ]:
bins = np.arange(16, 23.01, 0.25)
mid = 0.5 * (bins[1:] + bins[:-1])
med = {b: np.array([np.nanmedian(cat[f'psfmagerr_{b}'][(cat[f'psfmag_{b}'] >= lo) & (cat[f'psfmag_{b}'] < hi)])
                    for lo, hi in zip(bins[:-1], bins[1:])]) for b in 'gr'}
depth = {b: np.interp(np.log(0.1086), np.log(med[b]), mid) for b in 'gr'}   # sigma rises monotonically with mag

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(cat.psfmag_g, bins, histtype='step', lw=2, label='g')
ax[0].hist(cat.psfmag_r, bins, histtype='step', lw=2, label='r')
ax[0].set(yscale='log', xlabel='PSF magnitude', ylabel='point sources per 0.25 mag', title='number counts')
ax[0].legend()
for b in 'gr':
    ax[1].plot(mid, med[b], 'o-', label=f'{b}: S/N = 10 at {depth[b]:.2f}')
ax[1].axhline(0.1086, c='k', lw=0.8, ls='--')
ax[1].set(yscale='log', xlabel='PSF magnitude', ylabel='median magnitude error', title='photometric errors')
ax[1].legend()
plt.tight_layout()
print({b: round(float(v), 2) for b, v in depth.items()})
assert 21.5 < depth['g'] < 23.0

**Figure 0.1.** Left: number counts of point sources. The counts keep rising to g ≈ 22 and then turn over as
the survey becomes incomplete; the r-band counts stop early only because we selected g < 23. Right: median
magnitude error. The 10σ depth of SDSS is about g ≈ 22.

## 0.4 Is the patch fully observed?

A star-count map is only meaningful if we know where the survey looked. `bootes1_coverage` counts all primary
objects (any type, r < 22) in 0.1° × 0.1° cells; with ~10⁴ such objects per square degree, an observed cell
holds ~100 of them and an empty cell means "not observed".

In [ ]:
cov = load('bootes1_coverage')
cra, cdec = (cov.ira + 0.5) * 0.1, (cov.idec + 0.5) * 0.1
inside = np.hypot(*tangent(cra, cdec, ra0, dec0)) < 1.9          # cells well inside the 2-degree patch
area_cells = np.hypot(*tangent(*np.meshgrid(np.arange(205, 215, 0.1) + 0.05, np.arange(10, 19, 0.1) + 0.05),
                               ra0, dec0)) < 1.9                  # all cells of the same region, observed or not
frac = (inside & (cov.n > 20)).sum() / area_cells.sum()
print(f'cells observed inside 1.9 deg: {frac:.3f}; median objects per cell: {cov.n[inside].median():.0f}')
assert frac > 0.99

## 0.5 A first look at Boötes I

Boötes I lies at a distance modulus $\mu = 19.11$ (66 kpc) and has a half-light radius $r_h \approx 10'$ (LVDB).
In the map of *all* point sources it is hard to see. Its stars are old and metal-poor, so its red-giant
branch has $0.2 < (g-r)_0 < 0.7$, and at this distance its main-sequence turn-off lies at g ≈ 23, below our limit.
Selecting faint, bluish stars already makes it stand out.

In [ ]:
xi, eta = tangent(cat.ra, cat.dec, ra0, dec0)
R = np.hypot(xi, eta) * 60                                       # arcmin from the centre
sel = (gr0 > 0.0) & (gr0 < 0.7) & (g0 > 19) & (g0 < 22.5)

fig, ax = plt.subplots(1, 3, figsize=(16, 4.8))
e = np.linspace(-2, 2, 81)
for a, s, t in [(ax[0], np.ones(len(cat), bool), 'all point sources'), (ax[1], sel, r'19 < $g_0$ < 22.5, 0 < $(g-r)_0$ < 0.7')]:
    h = a.hist2d(xi[s], eta[s], bins=[e, e], cmap='Greys')[3]
    a.set(xlabel=r'$\xi$ [deg]', ylabel=r'$\eta$ [deg]', title=t, aspect='equal')
    a.invert_xaxis()                                              # east to the left, as on the sky
    plt.colorbar(h, ax=a, label='stars per 3′ × 3′ pixel')
rh = lvdb.loc['bootes_1', 'rhalf']
on, off = R < rh, (R > 60) & (R < 120)
scale = rh**2 / (120**2 - 60**2)                                 # ratio of the two areas
ax[2].scatter(gr0[off], g0[off], s=1, c='0.75', label='60′–120′ (all)')
ax[2].scatter(gr0[on], g0[on], s=4, c='C3', label=f'R < r_h = {rh:.1f}′')
ax[2].set(xlim=(-0.5, 1.5), ylim=(23, 16), xlabel=r'$(g-r)_0$', ylabel=r'$g_0$', title='colour–magnitude diagram')
ax[2].legend(markerscale=4)
plt.tight_layout()

n_on, n_bg = sel[on].sum(), sel[off].sum() * scale
print(f'selected stars inside r_h: {n_on}; expected from the background: {n_bg:.1f}; ratio {n_on / n_bg:.1f}')
assert n_on > 2 * n_bg

**Figure 0.2.** Left: all 59 000 point sources around Boötes I — the satellite is invisible. Middle: the same
stars restricted to faint, bluish colours — a compact overdensity appears at the centre. Right: the
colour–magnitude diagram (CMD) inside $r_h$ (red) on top of an outer annulus (grey); the red points trace a
red-giant branch and a blue horizontal branch at $g_0 \approx 19.6$ that the field does not have.

This is the whole idea of this tutorial: a satellite is an excess of stars that are *close together on the sky
and share a stellar population*. The rest of the tutorial turns this picture into a quantitative search, and
asks the critical question first: are the objects we call "stars" really stars? (Chapter 1).

## Summary

- The data are small sky patches extracted from WSDB with q3c cone and polygon queries; the manifest records each query.
- SDSS point sources reach S/N = 10 at g ≈ 22.
- Extinction is stored per object; dereddened magnitudes are $m_0 = m - A_m$.
- Boötes I appears as an excess of faint, blue stars inside its half-light radius.